# Fase 3: Programación orientada a objetos

Se utilizará como referencia el dataset de F2, correspondiente a licitaciones adjudicadas y desiertas del primer semestre de 2026, el objetivo es construir componentes reutilizables y medir su eficiencia.

In [ ]:
from pathlib import Path
import pandas as pd

raiz = Path.cwd().resolve().parents[1]

# Carpeta oficial de datos procesados (Paso 8 de la corrección)
carpeta_f2 = raiz / "F2" / "data" / "processed"
if not (carpeta_f2 / "licitaciones_filtradas_provisional.csv.gz").is_file():
    carpeta_f2 = raiz / "F2" / "notebooks" / "resultados_f2"

print("Raíz del proyecto:", raiz)
print("¿Existe la carpeta de F2?:", carpeta_f2.is_dir())

archivos = [
    "licitaciones_filtradas_provisional.csv.gz",
    "matriz_filtrada_provisional.csv.gz",
    "registros_pendientes_revision.csv",
    "resumen_final_f2.csv"
]

for archivo in archivos:
    ruta = carpeta_f2 / archivo
    print(f"{archivo}: {ruta.is_file()}")

Raíz del proyecto: C:\Users\nacho\proyecto-grupo_1-mcdi500
¿Existe la carpeta de F2?: True
licitaciones_filtradas_provisional.csv.gz: True
matriz_filtrada_provisional.csv.gz: True
registros_pendientes_revision.csv: True
resumen_final_f2.csv: True


In [2]:

# Cargamos el conjunto analítico de F2
df_licitaciones_f2 = pd.read_csv(
    carpeta_f2 / "licitaciones_filtradas_provisional.csv.gz"
)

df_matriz_f2 = pd.read_csv(
    carpeta_f2 / "matriz_filtrada_provisional.csv.gz"
)

print("Licitaciones F2:", df_licitaciones_f2.shape)
print("Matriz F2:", df_matriz_f2.shape)
print("Códigos únicos:",
      df_licitaciones_f2["CodigoExterno"].nunique())
print("Códigos duplicados:",
      df_licitaciones_f2["CodigoExterno"].duplicated().sum())

print("\nDistribución de EsDesierta:")
print(df_licitaciones_f2["EsDesierta"].value_counts())

assert len(df_licitaciones_f2) == 41607
assert len(df_matriz_f2) == 41607
assert df_licitaciones_f2["CodigoExterno"].is_unique

print("\nValidación inicial de F3 correcta")

Licitaciones F2: (41607, 18)
Matriz F2: (41607, 40)
Códigos únicos: 41607
Códigos duplicados: 0

Distribución de EsDesierta:
EsDesierta
0    35622
1     5985
Name: count, dtype: int64

Validación inicial de F3 correcta


Despúes de la comprobación traeremos lo procesado en F2 mediante funciones independientes. Primero vamos a cargar los datos.

In [3]:

import sys
ruta_modulos = str(raiz / "F3" / "src")

if ruta_modulos not in sys.path:
    sys.path.insert(0, ruta_modulos)
    
from carga import cargar_datos

df_cargado = cargar_datos(
    carpeta_f2 / "licitaciones_filtradas_provisional.csv.gz"
)

# Con esto buscamos comprobamos que la nueva carga conserva el resultado
pd.testing.assert_frame_equal(
    df_cargado,
    df_licitaciones_f2
)

print("La función de carga conserva los datos de F2.")

Cargado: 41607 filas y 18 columnas
La función de carga conserva los datos de F2.


### Creación de la variable objetivo

Creamos una función para generar la variable binaria EsDesierta, con esta vamos distinguir entre licitaciones adjudicadas (0) y desiertas (1).

In [4]:
from preparacion import crear_variable_objetivo

df_entrada = df_licitaciones_f2.drop(
    columns=["EsDesierta"]
).copy()

df_objetivo = crear_variable_objetivo(df_entrada)

pd.testing.assert_series_equal(
    df_objetivo["EsDesierta"],
    df_licitaciones_f2["EsDesierta"]
)

print("La variable objetivo coincide con F2")

print("\nDistribución de licitaciones:")
print(df_objetivo["EsDesierta"].value_counts())

La variable objetivo coincide con F2

Distribución de licitaciones:
EsDesierta
0    35622
1     5985
Name: count, dtype: int64


### Cálculo de los días entre publicación y cierre de las licitacions

Se implementa una función para calcular la diferencia en días entre las fechas de publicación y cierre de cada licitación.

In [5]:
import importlib
import preparacion

importlib.reload(preparacion)

# Lalmamos a la función
from preparacion import calcular_dias_publicacion_cierre

df_entrada_fechas = df_licitaciones_f2.drop(
    columns=["DiasPublicacionCierre"]
).copy()

df_fechas = calcular_dias_publicacion_cierre(
    df_entrada_fechas
)

# Comparamos el resultado con F2 para saber que se conservan los rsultados
pd.testing.assert_series_equal(
    df_fechas["DiasPublicacionCierre"],
    df_licitaciones_f2["DiasPublicacionCierre"],
    check_exact=False,
    rtol=1e-10,
    atol=1e-10
)

print("El cálculo de días coincide con F2")

print("\nDuraciones negativas:",
      (df_fechas["DiasPublicacionCierre"] < 0).sum())

print("Duraciones faltantes:",
      df_fechas["DiasPublicacionCierre"].isna().sum())

print("\nPrimeros cinco resultados:")
display(
    df_fechas[
        ["CodigoExterno", "DiasPublicacionCierre"]
    ].head()
)

El cálculo de días coincide con F2

Duraciones negativas: 0
Duraciones faltantes: 0

Primeros cinco resultados:


,CodigoExterno,DiasPublicacionCierre
0,799512-11-LE25,14.0
1,2408-54-LP25,35.0
2,743848-1-LE25,26.0
3,1641-157-LR25,30.0
4,1058095-67-LE25,10.0


### Clase Transformador

Se implementa la base que establece una estructura común para las transformaciones de datos.

In [6]:
from transformadores import Transformador

transformador = Transformador("MontoEstimadoNumerico")

print("Columna seleccionada:", transformador.columna)
print("¿Está ajustado?:", transformador._ajustado)

try:
    transformador.transformar(df_licitaciones_f2)

except RuntimeError as error:
    print("Validación correcta:", error)

Columna seleccionada: MontoEstimadoNumerico
¿Está ajustado?: False
Validación correcta: Primero debe ejecutar ajustar()


### Clase ImputadorMediana

Se implementa la clase `ImputadorMediana`, que hereda transformador y permite tratar los valores faltantes de la variable MontoEstimadoNumerico.

In [7]:
import importlib
import transformadores

importlib.reload(transformadores)

from transformadores import ImputadorMediana

# Preparar los datos sin las columnas de imputación
df_entrada_montos = df_licitaciones_f2.drop(
    columns=[
        "MontoEstimadoImputado",
        "MontoFueImputado"
    ]
).copy()

# Crear y ajustar el imputador
imputador = ImputadorMediana("MontoEstimadoNumerico")

imputador.ajustar(df_entrada_montos)

# Aplicar la transformación
df_montos_f3 = imputador.transformar(
    df_entrada_montos
)

# Comparar los montos imputados con F2
pd.testing.assert_series_equal(
    df_montos_f3["MontoEstimadoImputado"],
    df_licitaciones_f2["MontoEstimadoImputado"]
)

# Comparar los indicadores de imputación con F2
pd.testing.assert_series_equal(
    df_montos_f3["MontoFueImputado"],
    df_licitaciones_f2["MontoFueImputado"]
)

print("La imputación coincide con F2")

print("\nMediana CLP utilizada:",
      imputador._parametros["mediana"])

print("Montos imputados:",
      df_montos_f3["MontoFueImputado"].sum())

print("Nulos restantes:",
      df_montos_f3["MontoEstimadoImputado"].isna().sum())

La imputación coincide con F2

Mediana CLP utilizada: 27600000.0
Montos imputados: 7
Nulos restantes: 0


### Codificación de variables categóricas

In [8]:
import importlib
import transformadores

importlib.reload(transformadores)

from transformadores import CodificadorNominal

columnas_categoricas = [
    "sector",
    "RegionUnidad",
    "Tipo"
]

df_codificado_f3 = df_licitaciones_f2.copy()

for columna in columnas_categoricas:

    codificador = CodificadorNominal(columna)

    df_codificado_f3 = (
        codificador.ajustar(df_codificado_f3)
        .transformar(df_codificado_f3)
    )

    print(
        columna,
        "- categorías:",
        len(codificador._parametros["categorias"])
    )

# Identificamos las columnas one-ht que estaban em f2
prefijos = tuple(
    columna + "_"
    for columna in columnas_categoricas
)

columnas_f2 = [
    columna for columna in df_matriz_f2.columns
    if columna.startswith(prefijos)
]

# columnas one-hot de f3
columnas_f3 = [
    columna for columna in df_codificado_f3.columns
    if columna.startswith(prefijos)
]

print("\nColumnas One-Hot en F2:", len(columnas_f2))
print("Columnas One-Hot en F3:", len(columnas_f3))

assert columnas_f3 == columnas_f2, (
    "Las columnas generadas no coinciden con F2"
)

pd.testing.assert_frame_equal(
    df_codificado_f3[columnas_f3],
    df_matriz_f2[columnas_f2]
)

print("\nLa codificación coincide con F2")

sector - categorías: 8
RegionUnidad - categorías: 16
Tipo - categorías: 12

Columnas One-Hot en F2: 36
Columnas One-Hot en F3: 36

La codificación coincide con F2


### Pipeline de transformaciones

In [9]:
from pipeline import Pipeline
from transformadores import ImputadorMediana, CodificadorNominal

# definimos los datos de entrada
df_entrada_pipeline = df_licitaciones_f2.drop(
    columns=[
        "MontoEstimadoImputado",
        "MontoFueImputado"
    ]
).copy()

pasos = [
    ImputadorMediana("MontoEstimadoNumerico"),
    CodificadorNominal("sector"),
    CodificadorNominal("RegionUnidad"),
    CodificadorNominal("Tipo")
]

# le damos un orden
pipeline_f3 = Pipeline(pasos)

pipeline_f3.ajustar(df_entrada_pipeline)

df_resultado_f3 = pipeline_f3.transformar(
    df_entrada_pipeline
)

columnas_matriz = df_matriz_f2.columns.tolist()

assert all(
    columna in df_resultado_f3.columns
    for columna in columnas_matriz
), "Faltan columnas de la matriz original"

df_matriz_f3 = df_resultado_f3[
    columnas_matriz
].copy()

pd.testing.assert_frame_equal(
    df_matriz_f3,
    df_matriz_f2
)

print("El pipeline completo coincide con F2")

print("Filas:", df_matriz_f3.shape[0])
print("Columnas:", df_matriz_f3.shape[1])

print("\nValores faltantes:",
      df_matriz_f3.isna().sum().sum())

El pipeline completo coincide con F2
Filas: 41607
Columnas: 40

Valores faltantes: 0


### Validación del código

In [10]:
# Caso normal

pd.testing.assert_frame_equal(
    df_matriz_f3,
    df_matriz_f2
)

assert len(df_matriz_f3) == 41607
assert df_matriz_f3.shape[1] == 40
assert df_matriz_f3.isna().sum().sum() == 0

print("Caso normal: validación correcta")
print("Filas:", len(df_matriz_f3))
print("Columnas:", df_matriz_f3.shape[1])

Caso normal: validación correcta
Filas: 41607
Columnas: 40


In [11]:
# Caso limite:

datos_limite = pd.DataFrame({
    "MontoEstimadoNumerico": [100000.0],
    "Moneda Adquisicion": ["Peso Chileno"],
    "Tipo": ["LE"]
})

imputador_limite = ImputadorMediana(
    "MontoEstimadoNumerico"
)

resultado_limite = (
    imputador_limite.ajustar(datos_limite)
    .transformar(datos_limite)
)

assert resultado_limite["MontoEstimadoImputado"].iloc[0] == 100000.0
assert resultado_limite["MontoFueImputado"].sum() == 0

codificador_limite = CodificadorNominal("Tipo")

resultado_limite = (
    codificador_limite.ajustar(resultado_limite)
    .transformar(resultado_limite)
)

assert "Tipo_LE" in resultado_limite.columns
assert resultado_limite["Tipo_LE"].iloc[0] == 1

print("Caso límite: validación correcta")
print("Monto imputado:", resultado_limite["MontoEstimadoImputado"].iloc[0])
print("Registros imputados:", resultado_limite["MontoFueImputado"].sum())
print("Categoría codificada:", resultado_limite["Tipo_LE"].iloc[0])

Caso límite: validación correcta
Monto imputado: 100000.0
Registros imputados: 0
Categoría codificada: 1


In [12]:
# Excepción:

imputador_error = ImputadorMediana(
    "MontoEstimadoNumerico"
)

try:
    imputador_error.transformar(datos_limite)

except RuntimeError as error:
    print("Excepción detectada correctamente:")
    print(error)

else:
    raise AssertionError(
        "La transformación debía generar un error"
    )

print("Caso de excepción: validación correcta")

Excepción detectada correctamente:
Primero debe ejecutar ajustar()
Caso de excepción: validación correcta


### Eficiencia del código

Con esto comparamos la eficiencia de nuestro código, vamos a medir dos operaciones de nuestro código primero vamos a medir la función `apply()` para evaluar individualmente los estados de las licitaciones (adjudicada, desierta, etc) versus la segunda implementación que utiliza netamente el código de F2

In [13]:
import timeit
import importlib
import medicion

importlib.reload(medicion)

from medicion import (
    version_con_apply,
    version_vectorizada,
    medir_memoria
)

resultado_a = version_con_apply(df_licitaciones_f2)
resultado_b = version_vectorizada(df_licitaciones_f2)

pd.testing.assert_series_equal(
    resultado_a,
    resultado_b
)

pd.testing.assert_series_equal(
    resultado_b,
    df_licitaciones_f2["EsDesierta"]
)

print("Ambas implementaciones coinciden con F2")

tiempo_a = min(
    timeit.timeit(
        lambda: version_con_apply(df_licitaciones_f2),
        number=10
    )
    for _ in range(5)
) / 10

tiempo_b = min(
    timeit.timeit(
        lambda: version_vectorizada(df_licitaciones_f2),
        number=10
    )
    for _ in range(5)
) / 10

memoria_a = medir_memoria(
    version_con_apply,
    df_licitaciones_f2
)

memoria_b = medir_memoria(
    version_vectorizada,
    df_licitaciones_f2
)

comparacion = pd.DataFrame({
    "Implementacion": [
        "Apply",
        "Vectorizada"
    ],
    "Tiempo (segundos)": [
        tiempo_a,
        tiempo_b
    ],
    "Memoria maxima (MB)": [
        memoria_a,
        memoria_b
    ]
})

display(comparacion)

print("\nVersión más rápida:",
      "Apply" if tiempo_a < tiempo_b else "Vectorizada")

Ambas implementaciones coinciden con F2


,Implementacion,Tiempo (segundos),Memoria maxima (MB)
0,Apply,0.024508,3.257339
1,Vectorizada,0.001767,0.477572



Versión más rápida: Vectorizada


### Patrón de diseño

Se elige el patrón Factory, esto debido a que tenemos dos clases hijas y así centramos la creación de los objetos que seleccionamos en el pipeline del código.

In [14]:
from fabrica import crear_transformador

# Definimos las transformaciones
configuracion = [
    ("monto", "MontoEstimadoNumerico"),
    ("categorica", "sector"),
    ("categorica", "RegionUnidad"),
    ("categorica", "Tipo")
]

# aplicamos el factory
pasos_fabrica = []

for tipo, columna in configuracion:
    paso = crear_transformador(tipo, columna)
    pasos_fabrica.append(paso)

    print(columna, ":", type(paso).__name__)

pipeline_fabrica = Pipeline(pasos_fabrica)

pipeline_fabrica.ajustar(df_entrada_pipeline)

df_resultado_fabrica = pipeline_fabrica.transformar(
    df_entrada_pipeline
)

df_matriz_fabrica = df_resultado_fabrica[
    df_matriz_f2.columns
].copy()

pd.testing.assert_frame_equal(
    df_matriz_fabrica,
    df_matriz_f2
)

print("\nEl pipeline con Factory coincide con F2")
print("Filas:", df_matriz_fabrica.shape[0])
print("Columnas:", df_matriz_fabrica.shape[1])

MontoEstimadoNumerico : ImputadorMediana
sector : CodificadorNominal
RegionUnidad : CodificadorNominal
Tipo : CodificadorNominal

El pipeline con Factory coincide con F2
Filas: 41607
Columnas: 40


### Arquitectura

El código de F3 se organiza en módulos dentro de src:

- carga.py: con esto cargamos los archivos de datos.
- preparacion.py: este contiene funciones para preparar la variable objetivo.
- transformadores.py: definimos ImputadorMediana y CodificadorNominal.
- pipeline.py: muestra el plan de ejecución de las transformaciones.
- medicion.py: comparamos tiempo y memoria.
- fabrica.py: aplicación del factory.